# Logistic Regression 分类

## 本节目标

分类任务与回归任务的输出和 loss 不同。本节用一个线性分类器理解 logits、类别标签和交叉熵损失。

完成本节后，你应该能够：

- 说清本节主题解决了什么问题。
- 读懂核心 PyTorch API 的输入输出。
- 运行最小代码示例并解释结果。
- 修改实验参数并观察变化。

## 背景问题

本节关注的问题是：分类任务与回归任务的输出和 loss 不同。本节用一个线性分类器理解 logits、类别标签和交叉熵损失。

学习时建议先运行代码，再回头看解释。对初学者来说，能观察 shape、loss、accuracy 或可视化结果，比只记概念更重要。

## 核心概念

- 输入和输出的 shape 必须明确。
- loss 用来描述模型当前预测和目标之间的差距。
- 优化器根据梯度更新模型参数。
- 实验观察需要结合曲线、数值和样本可视化。
- 调试时优先检查 shape、dtype、学习率和训练/评估模式。

这里的关键不是堆公式，而是把概念落实到可运行代码。

## 最小代码示例：生成二维分类数据

这个代码块用于观察 `生成二维分类数据`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

x, y = make_classification(n_samples=500, n_features=2, n_redundant=0, n_informative=2, n_clusters_per_class=1, random_state=42)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler()
x_train = torch.tensor(scaler.fit_transform(x_train).astype(np.float32))
x_test = torch.tensor(scaler.transform(x_test).astype(np.float32))
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

plt.scatter(x_train[:, 0], x_train[:, 1], c=y_train, cmap="coolwarm", s=16)
plt.title("Classification data")
plt.grid(alpha=0.3)
plt.show()

## 最小代码示例：训练线性分类器

这个代码块用于观察 `训练线性分类器`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
model = nn.Linear(2, 2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
losses = []

for epoch in range(100):
    logits = model(x_train)
    loss = criterion(logits, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

with torch.no_grad():
    test_logits = model(x_test)
    acc = (test_logits.argmax(1) == y_test).float().mean().item()
print("test acc:", acc)

## 完整实验：查看概率

这个代码块用于观察 `查看概率`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
with torch.no_grad():
    probs = torch.softmax(test_logits[:5], dim=1)
print("logits:")
print(test_logits[:5])
print("probabilities:")
print(probs)

## 实验观察

`CrossEntropyLoss` 接收未归一化 logits，不需要提前 softmax。softmax 更适合在解释预测结果时使用。

从运行结果可以看到，代码输出不只是“能跑”，还可以帮助判断模型是否在按预期学习。建议把观察写进实验记录，而不是只保留最终准确率。

## 常见错误

- shape 不匹配：先打印输入、输出和标签 shape。
- dtype 不正确：分类标签通常是 `torch.long`，回归目标通常是 float。
- 忘记 `optimizer.zero_grad()`：梯度会累积。
- 学习率不合理：过大震荡，过小收敛慢。
- 评估阶段忘记 `model.eval()` 或 `torch.no_grad()`。

调试时可以先缩小数据集，确认模型能否在小样本上过拟合。

## 概念问答

**Q：为什么要从最小代码示例开始？**  
A：最小示例能隔离核心概念，降低同时排查多个问题的难度。

**Q：为什么每个实验都要看曲线或可视化？**  
A：单个最终数值不一定能解释训练过程，曲线和样本结果更容易暴露问题。

**Q：如果代码能运行但结果不好，先查什么？**  
A：优先检查数据、shape、label dtype、loss 使用方式和学习率。

## 延伸练习

- 把 `CrossEntropyLoss` 前手动 softmax，观察训练变化。
- 尝试更难的非线性数据。

这些练习不需要一次完成。建议每次只改一个变量，并记录改动前后的结果。

## 小结

本节把一个深度学习概念落到了可运行的 PyTorch 实验中。继续学习下一节前，建议确认自己能解释：输入是什么、模型做了什么、loss 如何计算、结果是否符合预期。